# Exploratory Data Analysis: Type 2 Diabetes Physiological Telemetry
### Happiest Health Digital Twin Challenge 2026

This notebook explores the multimodal physiological datasets for our virtual patient cohort:
1. **Static EHR Profiles**: Demographics, HbA1c, Fasting Glucose, BMI, Medications.
2. **Dynamic Sensor Stream**: 15-minute CGM readings, Heart Rate, Steps, Sleep, and Carbohydrate Ingestion.
3. **Coupled Physiological Dynamics**: Glucose velocity, exercise clearance, and postprandial excursions.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure data exists
from src.data.loader import load_patient_data

patient_profile, stream_df = load_patient_data(patient_id="PT-SYNTH-001")
print(f"Loaded Patient: {patient_profile['name']} ({patient_profile['patient_id']})")
print(f"Diagnosis: {patient_profile['diagnosis']} | HbA1c: {patient_profile['hba1c']}%")
print(f"Stream Shape: {stream_df.shape} (15-min intervals)")
stream_df.head()

## 1. Glucose Distribution & Glycemic Range
Evaluating time in range (TIR 70-140 mg/dL), time above range (TAR > 180 mg/dL), and time below range (TBR < 70 mg/dL).

In [ ]:
g = stream_df['glucose']
tir = np.mean((g >= 70) & (g <= 140)) * 100
tar = np.mean(g > 180) * 100
tbr = np.mean(g < 70) * 100

print(f"Time In Euglycemic Range (70-140 mg/dL): {tir:.1f}%")
print(f"Time Above Range (Hyperglycemia > 180 mg/dL): {tar:.1f}%")
print(f"Time Below Range (Hypoglycemia < 70 mg/dL): {tbr:.1f}%")
print(f"Mean Glucose: {g.mean():.1f} mg/dL | Std: {g.std():.1f} mg/dL | CV: {(g.std()/g.mean())*100:.1f}%")

## 2. Physiological Correlations (Glucose, Steps, HR, Meals)

In [ ]:
corr_cols = ['glucose', 'heart_rate', 'steps', 'sleep_hours', 'sleep_quality', 'carbs_intake']
corr_matrix = stream_df[corr_cols].corr()
print("Correlation Matrix:")
print(corr_matrix.round(3))

## 3. Visualizing 48-Hour Continuous Telemetry Stream

In [ ]:
# Plot first 192 ticks (48 hours)
subset = stream_df.iloc[:192]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
ax1.plot(subset.index, subset['glucose'], color='#2563EB', lw=2, label='CGM Glucose (mg/dL)')
ax1.axhline(180, color='r', ls='--', label='Hyperglycemic Threshold (180)')
ax1.axhspan(70, 140, color='g', alpha=0.15, label='Target Euglycemia')
ax1.set_ylabel('Glucose (mg/dL)')
ax1.legend(loc='upper right')
ax1.set_title('Continuous Interstitial Glucose Stream')

ax2.plot(subset.index, subset['heart_rate'], color='#E11D48', lw=1.5, label='Heart Rate (bpm)')
ax2.bar(subset.index, subset['steps'] / 20.0, color='#64748B', alpha=0.4, label='Steps / 20')
ax2.set_ylabel('HR (bpm) / Scaled Steps')
ax2.legend(loc='upper right')
ax2.set_xlabel('15-Minute Ticks')

plt.tight_layout()
plt.show()